# **EDA — Car Price Dataset**

## Objectives

* Perform exploratory data analysis on the cleaned car price dataset
* Identify correlations between car attributes and price
* Formulate and test business hypotheses
* Create visualisations to communicate key insights



## Inputs
* outputs/datasets/cleaned/car_prices_cleaned.csv

## Outputs

* Visualisations saved to outputs/figures/
* Hypothesis testing results

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2'

---

# Section 1 — Load Data

Loading the cleaned car price dataset that was prepared in the ETL pipeline notebook.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = pd.read_csv('outputs/datasets/cleaned/car_prices_cleaned.csv')
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,price_per_horsepower,price_per_enginesize
0,1,3,alfa-romero giulia,gas,std,2,convertible,rwd,front,88.6,...,3.47,2.68,9.0,111,5000,21,27,13495.0,121.576577,103.807692
1,2,3,alfa-romero stelvio,gas,std,2,convertible,rwd,front,88.6,...,3.47,2.68,9.0,111,5000,21,27,16500.0,148.648649,126.923077
2,3,1,alfa-romero Quadrifoglio,gas,std,2,hatchback,rwd,front,94.5,...,2.68,3.47,9.0,154,5000,19,26,16500.0,107.142857,108.552632
3,4,2,audi 100 ls,gas,std,4,sedan,fwd,front,99.8,...,3.19,3.40,10.0,102,5500,24,30,13950.0,136.764706,127.981651
4,5,2,audi 100ls,gas,std,4,sedan,4wd,front,99.4,...,3.19,3.40,8.0,115,5500,18,22,17450.0,151.739130,128.308824


## CarBrand Extraction

CarBrand is extracted from CarName here in order to test Hypothesis 2 
(luxury vs economy brands) and explore brand-level patterns throughout 
this notebook. The same extraction logic (including typo corrections) 
is also applied independently in 03_ML.ipynb during the ML pipeline, 
ensuring consistent transformations across both notebooks.

In [5]:
# Extract CarBrand from CarName (same logic as used consistently in 03_ML.ipynb)
df['CarBrand'] = df['CarName'].str.split().str[0].str.lower()
df['CarBrand'] = df['CarBrand'].replace({
    'maxda': 'mazda',
    'porcshce': 'porsche',
    'toyouta': 'toyota',
    'vokswagen': 'volkswagen',
    'vw': 'volkswagen'
})

---

## Statistical Concepts

Before analysing the data, here are the key statistical concepts used in this notebook:

**Mean** — the average value. Sensitive to outliers.
**Median** — the middle value when sorted. More robust to outliers than mean.
**Mode** — the most frequently occurring value.
**Standard Deviation (std)** — measures how spread out values are around the mean.
**Variance** — the square of standard deviation. Measures data dispersion.
**IQR (Interquartile Range)** — the range between Q1 (25th percentile) and Q3 
(75th percentile). Used to identify outliers.
**Skewness** — measures asymmetry of distribution. Positive skew = tail on right side.
**Normal Distribution** — bell-shaped, symmetric distribution where mean = median = mode.

*Note: The price variable in this dataset is NOT normally distributed 
(skewness = 1.78, mean $13,276 > median $10,295), which is common 
for price data with luxury outliers.*

---

# Section 2 — Business Hypotheses

1. Engine size and horsepower are the strongest predictors of car price
   — Understanding which technical features drive price is critical for 
   an automobile manufacturer entering the US market. If engine size is 
   the strongest predictor, the dataset suggests that consumers value 
   raw performance over other attributes.
   
   H0: Engine size and horsepower have no significant correlation with car price.
   H1: Engine size and horsepower are significantly correlated with car price.

2. Luxury car brands have significantly higher prices than economy brands
   — Brand positioning directly affects pricing strategy. A statistically 
   significant price gap in the data would confirm that brand perception 
   alone can justify higher pricing, independent of technical specifications.
   
   H0: There is no significant price difference between luxury and economy brands.
   H1: Luxury car brands have significantly higher prices than economy brands.

3. Diesel cars have higher prices than petrol (gas) cars
   — Fuel type affects both manufacturing cost and consumer demand. 
   However, with only 20 diesel cars vs 185 gas cars in the dataset, 
   this hypothesis may be limited by insufficient data.
   
   H0: There is no significant price difference between diesel and petrol cars.
   H1: Diesel cars have significantly higher prices than petrol cars.

4. Car body style significantly influences price
   — Body style affects manufacturing complexity and consumer appeal. 
   If convertibles are significantly more expensive than sedans in the data, 
   it suggests consumers pay a premium for aesthetics and exclusivity.
   
   H0: Car body style has no significant influence on price.
   H1: Car body style significantly influences car price.

---

# Section 3 — Hypothesis Testing

### Hypothesis 1: Engine size and horsepower are the strongest predictors of car price

### Plot 1 & 2 — Engine Size and Horsepower vs Price
The scatter plot below shows how car price changes as engine size increases. 
Each point represents one car. The trendline shows the overall direction.

In [6]:
fig = px.scatter(df, x='enginesize', y='price',
                 title='Engine Size vs Price',
                 labels={'enginesize': 'Engine Size', 'price': 'Price (USD)'},
                 hover_data=['CarName', 'CarBrand'],
                 trendline='ols')
fig.show()

fig2 = px.scatter(df, x='horsepower', y='price',
                  title='Horsepower vs Price',
                  labels={'horsepower': 'Horsepower', 'price': 'Price (USD)'},
                  hover_data=['CarName', 'CarBrand'],
                  trendline='ols')
fig2.show()

### Observation — Scatter Plots
Both charts show a clear positive trend — as engine size and horsepower 
increase, price tends to increase. The trendline confirms this pattern.
Hover over the points to explore individual car brands and models.

### Plot 3 — Which feature has the strongest correlation with price?
The bar chart shows the correlation between all numerical features and price. 
The longer the bar to the right, the stronger the positive relationship with price.
Engine size (0.87) tops the list, confirming our hypothesis.

In [7]:
num_cols = ['enginesize', 'horsepower', 'curbweight', 
            'carlength', 'carwidth', 'wheelbase',
            'citympg', 'highwaympg', 'boreratio', 'stroke']

price_corr = df[num_cols].corrwith(df['price']).sort_values()

fig = px.bar(x=price_corr.values,
             y=price_corr.index,
             orientation='h',
             title='Correlation of All Features with Price',
             labels={'x': 'Correlation with Price', 'y': 'Feature'},
             text=price_corr.values.round(2))
fig.show()

### Plot 4 — Relationship between Engine Size and Horsepower
Since both engine size and horsepower are strong predictors, this plot 
examines how they relate to each other. The trendline shows their correlation (0.81).

In [8]:
fig = px.scatter(df, x='enginesize', y='horsepower',
                 title='Engine Size vs Horsepower',
                 labels={'enginesize': 'Engine Size', 
                         'horsepower': 'Horsepower'},
                 hover_data=['CarBrand', 'CarName'],
                 trendline='ols',
                 color_discrete_sequence=['green'])
fig.show()

In [9]:
corr_value = df['enginesize'].corr(df['horsepower'])
print(f"Correlation between engine size and horsepower: {corr_value:.2f}")
print("This strong correlation (>0.8) confirms that larger engines produce more horsepower.")

Correlation between engine size and horsepower: 0.81
This strong correlation (>0.8) confirms that larger engines produce more horsepower.


### Conclusion — Hypothesis 1 partially confirmed 
Engine size (0.87) is confirmed as the strongest predictor of car price. However, curbweight (0.84) proved to be a stronger predictor than horsepower (0.81), which was unexpected. This suggests that the physical size and weight of a car influences price almost as much as engine performance.

Engine size and horsepower are also strongly correlated with each other (0.81), confirming that larger engines tend to produce more horsepower.

**Hypothesis 1 partially confirmed** — engine size is the strongest predictor, but curbweight unexpectedly outperformed horsepower.

---

### Hypothesis 2: Luxury car brands have significantly higher prices than economy brands
— Brand perception and reputation directly influences pricing

Brands with an average price above $20,000 are classified as luxury, 
all others as economy. The $20,000 threshold was selected as it represents 
approximately mean + 0.5 standard deviation ($13,276 + $7,988), separating 
the upper price segment from the majority of cars.

In [10]:
avg_price_brand = df.groupby('CarBrand')['price'].mean().sort_values(ascending=False).reset_index()

avg_price_brand['brand_type'] = avg_price_brand['price'].apply(
    lambda x: 'Luxury' if x > 20000 else 'Economy')

fig = px.bar(avg_price_brand, x='CarBrand', y='price',
             color='brand_type',
             title='Average Price by Car Brand',
             labels={'CarBrand': 'Car Brand', 'price': 'Average Price (USD)'},
             color_discrete_map={'Luxury': 'darkblue', 'Economy': 'lightblue'})
fig.show()

### Observation
The bar chart clearly shows a price separation between luxury and economy brands. 
Jaguar, Porsche and BMW have the highest average prices, confirming the hypothesis.
Buick also ranks in the luxury tier, which is somewhat unexpected for an American brand.
Notably, Alfa-Romero — traditionally considered a premium brand — falls below the 
$20,000 threshold in this dataset, suggesting their models here are entry-level variants.

### Additional Analysis — European vs American Brands
Since the bar chart revealed an interesting pattern, we explore whether 
there is a price difference between European and American car brands.

European brands (BMW, Porsche, Jaguar, Audi, Alfa-Romero, Volvo, Saab) 
are traditionally associated with premium engineering and design, while 
American brands (Buick, Mercury, Chevrolet, Dodge, Plymouth) are generally 
positioned as more accessible to the mass market.

This analysis was inspired by the observation that Buick — an American brand — 
was classified as luxury based on our $20,000 threshold, which raised the 
question of whether American and European brands follow different pricing patterns.

In [11]:
luxury_brands = avg_price_brand[avg_price_brand['brand_type'] == 'Luxury']['CarBrand'].tolist()
print("Luxury brands:", luxury_brands)

european = ['bmw', 'porsche', 'jaguar', 'audi', 'alfa-romero', 'volvo', 'saab']
american = ['buick', 'mercury', 'chevrolet', 'dodge', 'plymouth']

avg_price_brand['origin'] = avg_price_brand['CarBrand'].apply(
    lambda x: 'European' if x in european else ('American' if x in american else 'Other'))

fig = px.bar(avg_price_brand, x='CarBrand', y='price',
             color='origin',
             title='Average Price by Car Brand — European vs American vs Other',
             labels={'CarBrand': 'Car Brand', 'price': 'Average Price (USD)'})
fig.show()

Luxury brands: ['jaguar', 'buick', 'porsche', 'bmw']


### Sample size check
Before drawing conclusions from the origin comparison, we check how many cars 
each brand contributes to the dataset. Brands with very few observations can 
produce misleading average prices.

In [12]:
brand_counts = df['CarBrand'].value_counts().reset_index()
brand_counts.columns = ['CarBrand', 'count']
avg_price_brand = avg_price_brand.merge(brand_counts, on='CarBrand')
avg_price_brand[['CarBrand', 'price', 'origin', 'count']].sort_values('count')

,CarBrand,price,origin,count
6,mercury,16503.000000,American,1
14,renault,9595.000000,Other,2
0,jaguar,34600.000000,European,3
7,alfa-romero,15498.333333,European,3
21,chevrolet,6007.000000,American,3
16,isuzu,8916.500000,Other,4
2,porsche,31400.500000,European,5
9,saab,15223.333333,European,6
5,audi,17859.166714,European,7
19,plymouth,7963.428571,American,7


### Statistical Test — Independent Samples T-test (Parametric Test)
To confirm whether the price difference between luxury and economy brands 
is statistically significant, we run an independent samples t-test — a 
parametric test that assumes data follows a normal distribution.
A p-value below 0.05 means the difference is unlikely to be due to chance.

In [13]:
from scipy import stats

luxury = avg_price_brand[avg_price_brand['brand_type'] == 'Luxury']['price']
economy = avg_price_brand[avg_price_brand['brand_type'] == 'Economy']['price']

t_stat, p_value = stats.ttest_ind(luxury, economy)
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference — luxury brands are significantly more expensive")
else:
    print("Result: No statistically significant difference found")

T-statistic: 9.42
P-value: 0.0000
Result: Statistically significant difference — luxury brands are significantly more expensive


### Conclusion — Hypothesis 2 confirmed 

The analysis confirms that luxury brands have significantly higher prices 
than economy brands (t-test: p-value = 0.0000).

**Key findings:**
* Only 4 brands exceeded the $20,000 luxury threshold: Jaguar, Buick, Porsche and BMW
* European brands dominate the top price segment — Jaguar ($34.6k) and Porsche ($31.4k) 
  rank highest
* Buick ($33.6k) is a notable exception — the only American brand in the luxury tier, 
  ranking second highest overall (n=8, so not a data artefact)
* Alfa-Romero, Volvo, Audi and Saab — all traditionally premium European brands — 
  fall below the $20,000 threshold, suggesting only entry-level models are represented
* American brands (except Buick) cluster at the bottom: Chevrolet ($6k), Dodge ($7.9k)

**Important limitation:** Mercury has only n=1 car, making its average price unreliable. 
The t-test was run on brand average prices (n=22) rather than individual cars, 
which is a limitation — a larger sample would give more reliable results.

**Hypothesis 2 fully confirmed** — luxury brands are significantly more expensive, 
and European brands dominate the premium segment in this dataset.

----

### Hypothesis 3: Diesel cars have higher prices than petrol (gas) cars

## Dataset Balance Check

Before testing this hypothese, it is important to check whether the dataset 
is balanced across key categorical variable.

Fuel type distribution:
- Gas: 185 cars (90%)
- Diesel: 20 cars (10%)

This significant imbalance means hypothesis 3 (diesel vs gas pricing) 
may be affected by insufficient diesel data. To address this, we will 
use balanced sampling (20 gas vs 20 diesel) when running the statistical test.

In [14]:
df.filter(['fueltype'], axis=1).value_counts()

fueltype
gas         185
diesel       20
Name: count, dtype: int64

To test this hypothesis we compare the average price of diesel cars versus 
petrol (gas) cars in the dataset. We will then run a Parametic test to confirm 
whether any observed difference is statistically significant.

Note: The bar chart below is based on an imbalanced dataset (185 gas vs 20 diesel) 
and should be interpreted with caution.

In [15]:
avg_price_fuel = df.groupby('fueltype')['price'].mean().reset_index()

fig = px.bar(avg_price_fuel, x='fueltype', y='price',
             title='Average Price by Fuel Type',
             labels={'fueltype': 'Fuel Type', 'price': 'Average Price (USD)'},
             color='fueltype',
             color_discrete_map={'gas': 'lightblue', 'diesel': 'darkblue'})
fig.show()

### Note — Balanced Sampling
The original bar chart above is based on an imbalanced dataset (185 gas vs 20 diesel cars) 
and should be interpreted with caution. To create a fairer comparison, we use balanced 
sampling by randomly selecting 20 gas cars (random_state=42 ensures reproducibility) 
to match the 20 diesel cars available in the dataset.

In [16]:
gas_sample = df[df['fueltype'] == 'gas']['price'].sample(n=20, random_state=42)
diesel_sample = df[df['fueltype'] == 'diesel']['price']

balanced_df = pd.DataFrame({
    'price': pd.concat([diesel_sample, gas_sample]),
    'fueltype': ['diesel'] * 20 + ['gas'] * 20
})

avg_price_fuel_balanced = balanced_df.groupby('fueltype')['price'].mean().reset_index()

fig = px.bar(avg_price_fuel_balanced, x='fueltype', y='price',
             title='Average Price by Fuel Type (Balanced Sample — 20 vs 20)',
             labels={'fueltype': 'Fuel Type', 'price': 'Average Price (USD)'},
             color='fueltype',
             color_discrete_map={'gas': 'lightblue', 'diesel': 'darkblue'})
fig.show()

### T-test — Original (Imbalanced: 185 gas vs 20 diesel)
Note: This test is based on the imbalanced dataset and may not be reliable.

In [17]:
diesel = df[df['fueltype'] == 'diesel']['price']
gas = df[df['fueltype'] == 'gas']['price']

t_stat, p_value = stats.ttest_ind(diesel, gas)
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference — diesel cars are significantly more expensive")
else:
    print("Result: No statistically significant difference found")

T-statistic: 1.51
P-value: 0.1315
Result: No statistically significant difference found


### T-test — Balanced Sample (20 gas vs 20 diesel)
A more reliable test using equal sample sizes.

In [18]:
diesel = df[df['fueltype'] == 'diesel']['price']
gas = df[df['fueltype'] == 'gas']['price'].sample(n=20, random_state=42)

t_stat, p_value = stats.ttest_ind(diesel, gas)
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference — diesel cars are significantly more expensive")
else:
    print("Result: No statistically significant difference found")

T-statistic: -0.17
P-value: 0.8659
Result: No statistically significant difference found


### Conclusion — Hypothesis 3 rejected 
The original bar chart suggested diesel cars were more expensive, however 
this was misleading due to the heavily imbalanced dataset (185 gas vs 20 diesel).

With balanced sampling (20 vs 20), the chart actually shows gas cars have 
a higher average price. The t-test confirms there is no statistically 
significant difference (p-value = 0.8659, well above 0.05).

**Important limitation:** With only 20 diesel cars in the dataset, 
this hypothesis cannot be reliably tested. A larger and more balanced 
dataset would be needed to draw meaningful conclusions about diesel vs 
petrol pricing in the US market.

---

## Hypothesis 4: Car body style significantly influences price
— Convertibles and hardtops are expected to be more expensive than sedans and hatchbacks

To test this hypothesis we compare average prices across different car body styles 
using a box plot and a ANOVA test to confirm statistical significance.

In [19]:
fig = px.box(df, x='carbody', y='price',
             title='Price Distribution by Car Body Style',
             labels={'carbody': 'Body Style', 'price': 'Price (USD)'},
             color='carbody')
fig.show()

### Statistical Test — ANOVA (Parametric Test)
Since we are comparing more than two groups (5 body styles), we use ANOVA 
instead of a t-test. ANOVA tests whether at least one group has a 
significantly different mean price from the others.
A p-value below 0.05 means the differences are statistically significant.

In [20]:
convertible = df[df['carbody'] == 'convertible']['price']
hatchback = df[df['carbody'] == 'hatchback']['price']
sedan = df[df['carbody'] == 'sedan']['price']
wagon = df[df['carbody'] == 'wagon']['price']
hardtop = df[df['carbody'] == 'hardtop']['price']

f_stat, p_value = stats.f_oneway(convertible, hatchback, sedan, wagon, hardtop)
print(f"F-statistic: {f_stat:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference between body styles")
else:
    print("Result: No statistically significant difference found")

F-statistic: 8.03
P-value: 0.0000
Result: Statistically significant difference between body styles


### Conclusion — Hypothesis 4 confirmed 
The ANOVA test confirms a statistically significant price difference between 
car body styles (p-value = 0.0000, well below the 0.05 threshold).

The box plot shows that convertibles have the highest median price, 
while hatchbacks are the most affordable body style. Hardtops show 
the largest price variation.

The hypothesis is confirmed — car body style significantly influences price.

---

# Conclusions and Next Steps

## EDA Summary
* 4 business hypotheses tested using parametric tests (t-test and ANOVA)
* Hypothesis 1 partially confirmed — engine size is the strongest predictor
* Hypothesis 2 confirmed — luxury brands are significantly more expensive
* Hypothesis 3 rejected — no significant price difference between diesel and gas
* Hypothesis 4 confirmed — body style significantly influences price

## CRISP-DM Reflection
This notebook covers the **Data Understanding** and **Modelling preparation** 
phases of CRISP-DM:

* **Data Understanding** — explored distributions, correlations and patterns
* **Hypothesis Testing** — used parametric tests (t-test, ANOVA) to validate 
  business hypotheses
* **Data Insights** — identified enginesize as strongest price predictor, 
  confirmed luxury brand premium, found dataset imbalance in fuel type
* **Limitations** — small dataset (205 rows), imbalanced fuel type distribution, 
  t-test run on brand averages rather than individual cars

Next: 03_ML.ipynb — Machine Learning model to predict car prices.